In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
print("Notebook siap!")
print(f"pandas versi: {pd.__version__}")
print(f"Numpy versi: {np.__version__}")

In [ ]:
# Load dataset Excel
df = pd.read_excel("../data/raw/Data Cuaca Harian SulawesiTenggara.xlsx")

# Lihat 5 baris pertama
df.head()

In [ ]:
# Berapa baris dan kolom?
print(f"Jumlah baris  : {df.shape[0]}")
print(f"Jumlah kolom  : {df.shape[1]}")
print(f"Nama kolom    : {df.columns.tolist()}")

In [ ]:
# Lihat tipe data tiap kolom
df.info()

In [ ]:
# Convert kolom Tanggal dari teks ke tanggal
df['Tanggal'] = pd.to_datetime(df['Tanggal'], format='%d-%m-%Y')

# Cek berhasil
print(f"Tipe data Tanggal sekarang: {df['Tanggal'].dtype}")
print(f"Tanggal paling awal : {df['Tanggal'].min()}")
print(f"Tanggal paling akhir: {df['Tanggal'].max()}")

In [ ]:
# Persentase data kosong per kolom
missing_persen = (df.isnull().sum() / len(df)) * 100
missing_persen = missing_persen.sort_values(ascending=False)
print(missing_persen[missing_persen > 0])

In [ ]:
# Isi data kosong dengan rata-rata kolom
kolom_angka = df.select_dtypes(include='number').columns
df[kolom_angka] = df[kolom_angka].fillna(df[kolom_angka].mean())

# Cek hasil - harusnya semua 0
print("Data kosong setelah dibersihkan:")
print(df.isnull().sum())

In [ ]:
# Statistik ringkas setelah dibersihkan
df.describe().round(2)

In [ ]:
# Lihat nama kolom persis (perhatikan karakter anehnya)
for i, col in enumerate(df.columns):
    print(f"{i}: '{col}'")

In [ ]:
# Rename kolom berdasarkan urutan (index) - paling aman
df.columns = [
    'tanggal',
    'suhu_min',
    'suhu_max',
    'suhu_rata',
    'kelembapan',
    'curah_hujan',
    'penyinaran',
    'kecepatan_angin'
]

# Cek hasil
print("Kolom baru:")
print(df.columns.tolist())
print()
df.head(3)

In [ ]:
# Cari hari paling panas & paling dingin
suhu_max_idx = df['suhu_rata'].idxmax()
suhu_min_idx = df['suhu_rata'].idxmin()

print("=== SUHU EKSTREM ===")
print(f"Hari paling panas : {df.loc[suhu_max_idx, 'tanggal'].date()}")
print(f"  Suhu rata-rata  : {df.loc[suhu_max_idx, 'suhu_rata']}°C")
print(f"  Suhu maksimum   : {df.loc[suhu_max_idx, 'suhu_max']}°C")
print()
print(f"Hari paling dingin: {df.loc[suhu_min_idx, 'tanggal'].date()}")
print(f"  Suhu rata-rata  : {df.loc[suhu_min_idx, 'suhu_rata']}°C")
print(f"  Suhu minimum    : {df.loc[suhu_min_idx, 'suhu_min']}°C")

In [ ]:
# Cari hari paling hujan
hujan_idx = df['curah_hujan'].idxmax()

print("=== CURAH HUJAN ===")
print(f"Hari paling hujan : {df.loc[hujan_idx, 'tanggal'].date()}")
print(f"Curah hujan       : {df.loc[hujan_idx, 'curah_hujan']} mm")
print()
print(f"Hari hujan (curah > 0) : {(df['curah_hujan'] > 0).sum()} hari")
print(f"Hari kering (curah = 0): {(df['curah_hujan'] == 0).sum()} hari")
print(f"Rata-rata curah hujan  : {df['curah_hujan'].mean():.2f} mm/hari")

In [ ]:
# Tambah kolom bulan
df['bulan'] = df['tanggal'].dt.to_period('M')

# Ringkasan per bulan
ringkasan = df.groupby('bulan').agg({
    'suhu_rata': 'mean',
    'curah_hujan': 'sum',
    'kelembapan': 'mean'
}).round(2)

ringkasan

In [ ]:
# Lihat distribusi curah hujan
print("=== STATISTIK CURAH HUJAN ===")
print(df['curah_hujan'].describe())
print()

# Cek nilai-nilai ekstrem
print("=== 10 NILAI CURAH HUJAN TERTINGGI ===")
print(df.nlargest(10, 'curah_hujan')[['tanggal', 'curah_hujan']])
print()

# Berapa hari yang curah hujannya > 500 mm (nggak realistis)
print("=== HARI DENGAN HUJAN EKSTREM (>500mm) ===")
print(f"Jumlah: {(df['curah_hujan'] > 500).sum()} hari dari {len(df)} hari")

In [ ]:
# Cek apakah ada nilai yang berulang (kemungkinan kode error)
print("=== 10 NILAI YANG PALING SERING MUNCUL ===")
print(df['curah_hujan'].value_counts().head(10))

In [ ]:
# Langkah 1: Ganti nilai 8888 dengan NaN (anggap sebagai data kosong)
df['curah_hujan'] = df['curah_hujan'].replace(8888.0, np.nan)

# Cek berapa yang jadi NaN sekarang
print(f"Jumlah data kosong curah_hujan setelah 8888 dihilangkan: {df['curah_hujan'].isnull().sum()}")
print()

# Langkah 2: Hitung rata-rata baru (tanpa 8888)
rata_baru = df['curah_hujan'].mean()
print(f"Rata-rata curah hujan yang baru: {rata_baru:.2f} mm/hari")
print(f"(Sebelumnya salah: 1034.49 mm/hari)")
print()

# Langkah 3: Isi NaN dengan rata-rata baru
df['curah_hujan'] = df['curah_hujan'].fillna(rata_baru)

# Langkah 4: Cek statistik setelah bersih
print("=== STATISTIK CURAH HUJAN SETELAH DIBERSIHKAN ===")
print(df['curah_hujan'].describe())

In [ ]:
# Cari hari paling hujan (setelah data bersih)
hujan_idx = df['curah_hujan'].idxmax()

print("=== CURAH HUJAN (SETELAH BERSIH) ===")
print(f"Hari paling hujan : {df.loc[hujan_idx, 'tanggal'].date()}")
print(f"Curah hujan       : {df.loc[hujan_idx, 'curah_hujan']:.2f} mm")
print()
print(f"Total hari hujan : {(df['curah_hujan'] > 0).sum()} hari")
print(f"Total hari kering: {(df['curah_hujan'] == 0).sum()} hari")
print(f"Rata-rata        : {df['curah_hujan'].mean():.2f} mm/hari")

In [ ]:
# Update ringkasan bulanan dengan data yang sudah bersih
ringkasan = df.groupby('bulan').agg({
    'suhu_rata': 'mean',
    'curah_hujan': 'sum',
    'kelembapan': 'mean'
}).round(2)

ringkasan

In [ ]:
# Ganti semua nilai > 500 mm jadi NaN (definitely outlier)
df.loc[df['curah_hujan'] > 500, 'curah_hujan'] = np.nan

# Cek berapa yang jadi NaN
print(f"Outlier yang dibersihkan: {df['curah_hujan'].isnull().sum()} nilai")
print()

# Isi NaN dengan rata-rata baru
rata_bersih = df['curah_hujan'].mean()
df['curah_hujan'] = df['curah_hujan'].fillna(rata_bersih)

print(f"Rata-rata final: {rata_bersih:.2f} mm/hari")
print()
print("=== STATISTIK FINAL ===")
print(df['curah_hujan'].describe())

In [ ]:
# Update ringkasan dengan data final
ringkasan = df.groupby('bulan').agg({
    'suhu_rata': 'mean',
    'curah_hujan': 'sum',
    'kelembapan': 'mean'
}).round(2)

ringkasan

In [ ]:
print("=== VERIFIKASI FINAL ===")
print(f"Curah hujan max        : {df['curah_hujan'].max():.2f} mm")
print(f"Curah hujan rata-rata  : {df['curah_hujan'].mean():.2f} mm/hari")
print(f"Bulan dengan hujan terbanyak:")
print(ringkasan['curah_hujan'].idxmax(), "→", ringkasan['curah_hujan'].max(), "mm")
print()
print(f"Bulan dengan hujan tersedikit:")
print(ringkasan['curah_hujan'].idxmin(), "→", ringkasan['curah_hujan'].min(), "mm")

In [ ]:
# Setup style grafik
plt.style.use('seaborn-v0_8-darkgrid')
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['font.size'] = 10

In [ ]:
fig, ax = plt.subplots()

ax.plot(df['tanggal'], df['suhu_rata'], color='orange', linewidth=0.8, label='Suhu Rata-rata')
ax.fill_between(df['tanggal'], df['suhu_min'], df['suhu_max'], 
                alpha=0.2, color='red', label='Range Suhu (Min-Max)')

ax.set_title('Suhu Harian Sulawesi Tenggara (2022-2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Tanggal')
ax.set_ylabel('Suhu (°C)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots()

warna = ['#e74c3c' if x > 300 else '#3498db' if x > 100 else '#f39c12' 
         for x in ringkasan['curah_hujan']]
bulan_str = ringkasan.index.astype(str)

ax.bar(bulan_str, ringkasan['curah_hujan'], color=warna, edgecolor='black', linewidth=0.5)

ax.set_title('Total Curah Hujan Bulanan', fontsize=14, fontweight='bold')
ax.set_xlabel('Bulan')
ax.set_ylabel('Curah Hujan (mm)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

In [ ]:
fig, ax1 = plt.subplots()

# Sumbu kiri: suhu
color1 = 'tab:red'
ax1.set_xlabel('Bulan')
ax1.set_ylabel('Suhu Rata-rata (°C)', color=color1)
ax1.plot(bulan_str, ringkasan['suhu_rata'], color=color1, marker='o', linewidth=2)
ax1.tick_params(axis='y', labelcolor=color1)
plt.xticks(rotation=45, ha='right')

# Sumbu kanan: kelembapan
ax2 = ax1.twinx()
color2 = 'tab:blue'
ax2.set_ylabel('Kelembapan (%)', color=color2)
ax2.plot(bulan_str, ringkasan['kelembapan'], color=color2, marker='s', linewidth=2)
ax2.tick_params(axis='y', labelcolor=color2)

plt.title('Suhu vs Kelembapan Bulanan', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
print("=== CEK SUHU EKSTREM ===")
print(f"suhu_min  max: {df['suhu_min'].max()}")
print(f"suhu_max  max: {df['suhu_max'].max()}")
print(f"suhu_rata max: {df['suhu_rata'].max()}")
print()

# Cari baris dengan suhu_max > 50 (pasti salah)
print("=== BARIS DENGAN SUHU ANEH ===")
aneh = df[(df['suhu_max'] > 50) | (df['suhu_min'] > 50) | (df['suhu_rata'] > 50)]
print(aneh[['tanggal', 'suhu_min', 'suhu_max', 'suhu_rata']])

In [ ]:
# Ganti suhu > 50°C atau < 10°C jadi NaN (pasti kode error)
kolom_suhu = ['suhu_min', 'suhu_max', 'suhu_rata']

for kol in kolom_suhu:
    # Ganti nilai di luar rentang wajar
    df.loc[(df[kol] > 50) | (df[kol] < 10), kol] = np.nan
    
    # Isi NaN dengan rata-rata yang sudah bersih
    rata = df[kol].mean()
    df[kol] = df[kol].fillna(rata)
    
    print(f"{kol}: setelah bersih, max={df[kol].max():.2f}, min={df[kol].min():.2f}")

print()
print("=== STATISTIK SUHU FINAL ===")
print(df[['suhu_min', 'suhu_max', 'suhu_rata']].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df['tanggal'], df['suhu_rata'], color='orange', linewidth=0.8, label='Suhu Rata-rata')
ax.fill_between(df['tanggal'], df['suhu_min'], df['suhu_max'], 
                alpha=0.2, color='red', label='Range Suhu (Min-Max)')

ax.set_title('Suhu Harian Sulawesi Tenggara (2022-2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Tanggal')
ax.set_ylabel('Suhu (°C)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Bersihkan suhu dengan metode IQR (lebih cerdas)
kolom_suhu = ['suhu_min', 'suhu_max', 'suhu_rata']

for kol in kolom_suhu:
    Q1 = df[kol].quantile(0.25)
    Q3 = df[kol].quantile(0.75)
    IQR = Q3 - Q1
    
    # Batas wajar: Q1 - 1.5*IQR sampai Q3 + 1.5*IQR
    batas_bawah = Q1 - 1.5 * IQR
    batas_atas = Q3 + 1.5 * IQR
    
    print(f"{kol}: batas wajar = {batas_bawah:.2f} - {batas_atas:.2f}")
    
    # Ganti outlier jadi NaN
    df.loc[(df[kol] < batas_bawah) | (df[kol] > batas_atas), kol] = np.nan
    
    # Isi dengan rata-rata
    rata = df[kol].mean()
    df[kol] = df[kol].fillna(rata)

print()
print("=== STATISTIK SUHU SETELAH IQR ===")
print(df[kolom_suhu].describe().round(2))

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

ax.plot(df['tanggal'], df['suhu_rata'], color='orange', linewidth=0.8, label='Suhu Rata-rata')
ax.fill_between(df['tanggal'], df['suhu_min'], df['suhu_max'], 
                alpha=0.2, color='red', label='Range Suhu (Min-Max)')

ax.set_title('Suhu Harian Sulawesi Tenggara (2022-2023)', fontsize=14, fontweight='bold')
ax.set_xlabel('Tanggal')
ax.set_ylabel('Suhu (°C)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Simpan dataset yang sudah bersih ke data/processed/
output_path = '../data/processed/cuaca_sultra_clean.csv'
df.to_csv(output_path, index=False)

print(f"✅ Dataset bersih tersimpan di: {output_path}")
print(f"   Jumlah baris : {len(df)}")
print(f"   Jumlah kolom : {df.shape[1]}")
print(f"   Kolom        : {df.columns.tolist()}")

## 📊 Kesimpulan

Dataset cuaca Sulawesi Tenggara (2022-2023) telah melalui proses:
1. ✅ Load dataset Excel (699 baris, 8 kolom)
2. ✅ Convert kolom tanggal ke datetime
3. ✅ Bersihkan missing values
4. ✅ Bersihkan outlier curah hujan (nilai 8888 & >500mm)
5. ✅ Bersihkan outlier suhu (nilai 9999 & metode IQR)
6. ✅ Visualisasi: suhu harian, curah hujan bulanan, suhu vs kelembapan

**Insight:**
- Curah hujan rata-rata: 7.75 mm/hari
- Suhu rata-rata: 26.98°C
- Bulan tertinggi curah hujan: Juli 2022 (464 mm)
- Terlihat anomali El Nino di akhir 2023 (suhu naik, hujan turun)